In [ ]:
import geopandas as gpd
import pandas as pd
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

In [ ]:
# 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

print("LIA筛选完成: {} 条".format(len(lia_matched)))

In [ ]:
# 筛选MainRGI对应的RGI冰川
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()

# 转WGS84
lia_matched = lia_matched.to_crs("EPSG:4326")
rgi_to_calibrate = rgi_to_calibrate.to_crs("EPSG:4326")

# 建立映射字典: LIA的UniqueID → MainRGI
lia_to_main_rgi = dict(zip(
    lia_matched['UniqueID'].astype(int),
    lia_matched['MainRGI']
))

print("RGI待校准: {} 条".format(len(rgi_to_calibrate)))
print("映射关系: {} 条".format(len(lia_to_main_rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, utils

WORKING_DIR = "/mnt/f/oggm_related/oggm_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_13/dh_13_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"
# 读取MB数据
dfs = []
for path in MB_PATHS:
    print("读取: {}".format(path))
    df = pd.read_csv(path)
    dfs.append(df)
mb_all = pd.concat(dfs, ignore_index=True)
mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')
print("MB数据: {} 条".format(len(mb_data)))
# 清理RGI数据
rgi = rgi_to_calibrate.copy()
rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print("有MB数据的RGI冰川: {}".format(len(rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, GlacierDirectory
cfg.initialize()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 26

gdirs_rgi = workflow.init_glacier_directories(rgi, reset=False)
print("初始化完成: {}".format(len(gdirs_rgi)))

In [ ]:
for gdir in gdirs_rgi:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_rgi)))

In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True

execute_entity_task(tasks.glacier_masks, gdirs_rgi)
execute_entity_task(tasks.compute_centerlines, gdirs_rgi)
execute_entity_task(tasks.initialize_flowlines, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_line, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_rgi)
execute_entity_task(tasks.catchment_area, gdirs_rgi)
execute_entity_task(tasks.catchment_intersections, gdirs_rgi)
execute_entity_task(tasks.catchment_width_geom, gdirs_rgi)
execute_entity_task(tasks.catchment_width_correction, gdirs_rgi)
print("预处理完成")

In [ ]:
cfg.PARAMS['baseline_climate'] = 'W5E5'
for gdir in gdirs_rgi:
    tasks.process_w5e5_data(gdir)#不需要了
# execute_entity_task(tasks.process_cru_data, gdirs_rgi)
print("气候数据处理完成")

In [ ]:
from oggm.core.massbalance import mb_calibration_from_scalar_mb

cfg.PARAMS['use_winter_prcp_fac'] = False
cfg.PARAMS['prcp_fac'] = None

success = 0
fail = 0
failed_list = []

for gdir in gdirs_rgi:
    try:
        rgi_id = gdir.rgi_id
        ref_mb = float(mb_data.loc[rgi_id, 'dmdtda_kg'])
        mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            overwrite_gdir=True,
            ref_mb_years=(2000, 2020),
            calibrate_param1='prcp_fac',   # 第一步：降水因子
            calibrate_param2='melt_f',     # 第二步：融化因子
            calibrate_param3='temp_bias',  # 第三步：温度偏差
        )
        success += 1
    except Exception as e:
        fail += 1
        failed_list.append((gdir.rgi_id, str(e)[:80]))
        print(f"失败: {gdir.rgi_id} - {str(e)[:60]}")

print(f"校准完成: 成功 {success}, 失败 {fail}")

In [ ]:
calibrated_rgi_ids = set([gdir.rgi_id for gdir in gdirs_rgi])
lia_main_rgis = set(lia_matched['MainRGI'].tolist())

missing = lia_main_rgis - calibrated_rgi_ids
print("缺失校准参数的MainRGI: {} 条".format(len(missing)))
print(missing)

In [ ]:
# 1. 准备RGI校准参数字典
import json
import os

rgi_params = {}
for gdir in gdirs_rgi:
    fpath = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fpath, 'r') as f:
        rgi_params[gdir.rgi_id] = json.load(f)

# 计算区域平均（给缺失的10条用）
import numpy as np
avg_melt_f = np.mean([p['melt_f'] for p in rgi_params.values()])
avg_prcp_fac = np.mean([p['prcp_fac'] for p in rgi_params.values()])
avg_temp_bias = np.mean([p['temp_bias'] for p in rgi_params.values()])

print("已校准RGI参数: {} 条".format(len(rgi_params)))
print("区域平均: melt_f={:.2f}, prcp_fac={:.2f}, temp_bias={:.2f}".format(
    avg_melt_f, avg_prcp_fac, avg_temp_bias))

In [ ]:
# 2. 建立LIA → mb参数的完整映射
lia_mb_params = {}
missing_count = 0

for idx, row in lia_matched.iterrows():
    unique_id = int(row['UniqueID'])
    main_rgi = row['MainRGI']

    if main_rgi in rgi_params:
        p = rgi_params[main_rgi]
        lia_mb_params[unique_id] = {
            'melt_f': p['melt_f'],
            'prcp_fac': p['prcp_fac'],
            'temp_bias': p['temp_bias'],
            'source': main_rgi,
        }
    else:
        lia_mb_params[unique_id] = {
            'melt_f': avg_melt_f,
            'prcp_fac': avg_prcp_fac,
            'temp_bias': avg_temp_bias,
            'source': 'regional_average',
        }
        missing_count += 1

print("LIA参数映射完成: {} 条".format(len(lia_mb_params)))
print("使用区域平均: {} 条".format(missing_count))

In [ ]:
# 3. 准备LIA冰川数据，初始化OGGM
from oggm import cfg, workflow, tasks

LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

# 为LIA创建标准RGI格式的ID
lia_for_oggm = lia_matched.copy()
lia_for_oggm['RGIId'] = lia_for_oggm['UniqueID'].apply(
    lambda x: "RGI60-15.{:05d}".format(int(x))
)

# 确保有Area字段
if 'Area' not in lia_for_oggm.columns:
    lia_for_oggm['Area'] = lia_for_oggm['LIAArea']

print("LIA冰川准备完成: {} 条".format(len(lia_for_oggm)))
print(lia_for_oggm[['RGIId', 'UniqueID', 'MainRGI', 'Area']].head())

In [ ]:
lia_sorted = lia_matched.sort_values('UniqueID').copy()
lia_cooked = utils.cook_rgidf(
    lia_sorted,
    o1_region='15',
    bgndate='16500101',
    ids=lia_sorted['UniqueID'].astype(int).tolist(),
    assign_column_values={'LIAArea': 'Area'}
)

lia_cooked['UniqueID'] = lia_sorted['UniqueID'].values
lia_cooked['MainRGI'] = lia_sorted['MainRGI'].values

print("cook完成: {} 条".format(len(lia_cooked)))
print(lia_cooked[['RGIId', 'UniqueID', 'Area']].head())

In [ ]:
LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_lia = workflow.init_glacier_directories(lia_cooked,reset= False)
print("LIA初始化完成: {} 条".format(len(gdirs_lia)))

In [ ]:
for gdir in gdirs_lia:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_lia)))

In [ ]:
from oggm.workflow import execute_entity_task

execute_entity_task(tasks.glacier_masks, gdirs_lia)
execute_entity_task(tasks.compute_centerlines, gdirs_lia)
execute_entity_task(tasks.initialize_flowlines, gdirs_lia)
execute_entity_task(tasks.compute_downstream_line, gdirs_lia)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_lia)
execute_entity_task(tasks.catchment_area, gdirs_lia)
execute_entity_task(tasks.catchment_intersections, gdirs_lia)
execute_entity_task(tasks.catchment_width_geom, gdirs_lia)
execute_entity_task(tasks.catchment_width_correction, gdirs_lia)
print("预处理完成: {}".format(len(gdirs_lia)))

In [ ]:
from tqdm import tqdm

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'


cfg.PARAMS['baseline_climate'] = 'CRU'
LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_lia:
    tasks.process_cru_data(gdir)

In [ ]:
from oggm.shop import gcm_climate

for gdir in gdirs_lia:
    gcm_climate.process_lmr_data(gdir)
print("LMR气候数据处理完成: {}".format(len(gdirs_lia)))

In [ ]:
import json
import os

success = 0
fail = 0

for gdir in gdirs_lia:
    try:
        # 从RGIId提取UniqueID
        unique_id = int(gdir.rgi_id.split('.')[-1])

        # 获取对应的mb参数
        params = lia_mb_params[unique_id]

        # 构建mb_calib.json
        mb_calib = {
            'rgi_id': gdir.rgi_id,
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
            'temp_bias': params['temp_bias'],
            'bias': 0,
            'reference_mb': 0,
            'reference_mb_err': 0,
            'reference_period': '1650-01-01_1650-01-01',
            'baseline_climate_source': 'LMR',
        }

        # 写入文件
        fpath = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fpath, 'w') as f:
            json.dump(mb_calib, f, indent=2)

        success += 1
    except Exception as e:
        fail += 1
        print("失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

print("mb_calib写入完成: 成功 {}, 失败 {}".format(success, fail))

execute_entity_task(tasks.prepare_for_inversion, gdirs_lia)
execute_entity_task(tasks.mass_conservation_inversion, gdirs_lia)
execute_entity_task(tasks.filter_inversion_output, gdirs_lia)

print("冰厚反演完成: {}".format(len(gdirs_lia)))

In [ ]:
# 筛选成功的gdir
gdirs_ok = [gdir for gdir in gdirs_lia if gdir.rgi_id != 'RGI60-15.01038']
print("继续处理: {} 条".format(len(gdirs_ok)))

# # 冰厚反演
# execute_entity_task(tasks.prepare_for_inversion, gdirs_ok)
# execute_entity_task(tasks.mass_conservation_inversion, gdirs_ok)
# execute_entity_task(tasks.filter_inversion_output, gdirs_ok)
#
# print("冰厚反演完成: {}".format(len(gdirs_ok)))

In [ ]:
execute_entity_task(tasks.init_present_time_glacier, gdirs_ok)
print("init_present_time_glacier完成: {}".format(len(gdirs_ok))) # 将反演结果转换为可运行的模型flowlines

In [ ]:
from oggm.core.flowline import run_constant_climate, FluxBasedModel
from oggm.core.massbalance import MonthlyTIModel
from functools import partial

mb_model_class = partial(MonthlyTIModel, check_calib_params=False)
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20
cfg.PARAMS['cfl_min_dt'] = 20
cfg.PARAMS['continue_on_error'] = True  #
mb_model_class = partial(MonthlyTIModel, check_calib_params=False)

workflow.execute_entity_task(
    run_constant_climate,
    gdirs_ok,
    nyears=300,
    y0=1650,
    halfsize=30,
    climate_filename='gcm_data',
    mb_model_class=mb_model_class,
    evolution_model=FluxBasedModel,
    store_model_geometry=True,
    output_filesuffix='_spinup_LIA',
)


In [ ]:
# 错误收集
import os
success_gdirs = []
failed_gdirs = []

for gdir in gdirs_ok:
    fp = gdir.get_filepath('model_diagnostics', filesuffix='_spinup_LIA')
    if os.path.exists(fp):
        success_gdirs.append(gdir.rgi_id)
    else:
        failed_gdirs.append(gdir.rgi_id)

print(f"成功: {len(success_gdirs)}, 失败: {len(failed_gdirs)}")
print(f"失败列表: {failed_gdirs}")

In [ ]:
from oggm.core.flowline import run_from_climate_data, FluxBasedModel
from oggm.workflow import execute_entity_task
from functools import partial
from oggm.core.massbalance import MonthlyTIModel

# 筛选成功spin-up的冰川
failed_spinup_ids = ['RGI60-15.00079', 'RGI60-15.00084', 'RGI60-15.00214', 'RGI60-15.01044',
                     'RGI60-15.01604', 'RGI60-15.01743', 'RGI60-15.01931', 'RGI60-15.01968',
                     'RGI60-15.02044', 'RGI60-15.02282', 'RGI60-15.02692', 'RGI60-15.02898',
                     'RGI60-15.02906', 'RGI60-15.04319', 'RGI60-15.05127', 'RGI60-15.05508']

gdirs_run = [gdir for gdir in gdirs_lia if gdir.rgi_id not in failed_spinup_ids]
print("正演冰川数量: {}".format(len(gdirs_run)))

mb_model_class = partial(MonthlyTIModel, check_calib_params=False)
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 26
cfg.PARAMS['cfl_min_dt'] = 60
cfg.PARAMS['continue_on_error'] = True

from oggm.core.massbalance import MonthlyTIModel

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

execute_entity_task(
    run_from_climate_data,
    gdirs_run,
    min_ys=1650,
    ye=2000,
    climate_filename='gcm_data',
    mb_model_class=MyMBModel,
    evolution_model=FluxBasedModel,
    init_model_filesuffix='_spinup_LIA',
    output_filesuffix='_LIA_forward'
)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

gdir = gdirs_run[150]  # 取第一个冰川

fp = gdir.get_filepath('gcm_data')
with xr.open_dataset(fp) as clim:
    # 先算年平均
    temp_annual = clim.temp.groupby('time.year').mean()

    # 10年滑动平均
    temp_10yr = temp_annual.rolling(year=30, center=True).mean()

    plt.figure(figsize=(12, 4))
    plt.plot(temp_annual.year, temp_annual.values, alpha=0.3, label='Annual')
    plt.plot(temp_10yr.year, temp_10yr.values, 'r-', linewidth=2, label='10-yr mean')
    plt.axvline(x=1450, color='g', linestyle='--', label='y0=1450')
    plt.axvline(x=1650, color='b', linestyle='--', label='min_ys=1650')
    plt.xlabel('Year')
    plt.ylabel('Temperature (K)')
    plt.title(f'LMR Temperature - {gdir.rgi_id}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 打印关键时段平均
    print("1440-1480 平均: {:.2f} K".format(float(temp_annual.sel(year=slice(1440,1480)).mean())))
    print("1650-1700 平均: {:.2f} K".format(float(temp_annual.sel(year=slice(1650,1700)).mean())))
    print("1850-1900 平均: {:.2f} K".format(float(temp_annual.sel(year=slice(1850,1900)).mean())))

In [ ]:
import json
import os
import numpy as np

temp_bias_list = []
melt_f_list = []
prcp_fac_list = []

for gdir in gdirs_run:
    try:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)

        temp_bias_list.append(params['temp_bias'])
        melt_f_list.append(params['melt_f'])
        prcp_fac_list.append(params['prcp_fac'])
    except:
        pass

print(f"统计冰川数: {len(temp_bias_list)}")
print()
print("=== temp_bias (K) ===")
print(f"  均值: {np.mean(temp_bias_list):.2f}")
print(f"  中位数: {np.median(temp_bias_list):.2f}")
print(f"  范围: {np.min(temp_bias_list):.2f} ~ {np.max(temp_bias_list):.2f}")
print(f"  标准差: {np.std(temp_bias_list):.2f}")

print()
print("=== melt_f (mm/K/day) ===")
print(f"  均值: {np.mean(melt_f_list):.2f}")
print(f"  中位数: {np.median(melt_f_list):.2f}")
print(f"  范围: {np.min(melt_f_list):.2f} ~ {np.max(melt_f_list):.2f}")
print(f"  标准差: {np.std(melt_f_list):.2f}")

print()
print("=== prcp_fac ===")
print(f"  均值: {np.mean(prcp_fac_list):.2f}")
print(f"  中位数: {np.median(prcp_fac_list):.2f}")
print(f"  范围: {np.min(prcp_fac_list):.2f} ~ {np.max(prcp_fac_list):.2f}")
print(f"  标准差: {np.std(prcp_fac_list):.2f}")

# 保存原始参数
original_tbs = {}
for gdir in gdirs_run:
    try:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)
        original_tbs[gdir.rgi_id] = params['temp_bias']
    except:
        pass

# 保存到文件
output_path = os.path.join(cfg.PATHS['working_dir'], 'original_temp_bias.json')
with open(output_path, 'w') as f:
    json.dump(original_tbs, f, indent=2)
print(f"已保存: {output_path}")
print(f"均值: {np.mean(list(original_tbs.values())):.2f}K")

In [ ]:
from oggm.core.massbalance import MonthlyTIModel
import numpy as np
import json
import os

gdir = gdirs_run[0]
fls = gdir.read_pickle('model_flowlines')
heights = fls[0].surface_h
fp = os.path.join(gdir.dir, 'mb_calib.json')

# 读原始参数
with open(fp, 'r') as f:
    params_orig = json.load(f)

original_tb = params_orig['temp_bias']

test_biases = [-3, -2, -1, 0, 1, 2]

for tb in test_biases:
    # 修改参数文件
    params_orig['temp_bias'] = tb
    with open(fp, 'w') as f:
        json.dump(params_orig, f)

    # 重新创建模型（这才会读新参数）
    mb_model = MonthlyTIModel(gdir, filename='gcm_data', check_calib_params=False)
    mb = mb_model.get_annual_mb(heights, year=1650)

    # 找ELA
    ela = np.interp(0, mb[::-1], heights[::-1])

    print(f"temp_bias={tb:+.0f}K → MB范围: {mb.min()*910:.0f}~{mb.max()*910:.0f} kg/m²/yr → ELA={ela:.0f}m")

# 恢复原参数
params_orig['temp_bias'] = original_tb
with open(fp, 'w') as f:
    json.dump(params_orig, f)

In [ ]:
import json
import os
import xarray as xr
import numpy as np
from oggm.core.flowline import run_constant_climate, FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import cfg

offsets = [0,-0.5, -1,-1.5, -2,-2.5, -3, -3.5, -4]
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 22
# 设置继续运行即使出错
cfg.PARAMS['continue_on_error'] = True

# 保存原始temp_bias
original_tbs = {}
valid_gdirs = []

for gdir in gdirs_run:
    try:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)
        original_tbs[gdir.rgi_id] = params['temp_bias']
        valid_gdirs.append(gdir)
    except:
        print(f"跳过: {gdir.rgi_id}")

print(f"有效冰川: {len(valid_gdirs)}")

results = {gdir.rgi_id: {'offset_changes': {}} for gdir in valid_gdirs}

for offset in offsets:
    print(f"\n=== 测试 offset={offset:+.1f}K ===")

    # 修改所有冰川的temp_bias
    for gdir in valid_gdirs:
        fp = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fp, 'r') as f:
            params = json.load(f)
        params['temp_bias'] = original_tbs[gdir.rgi_id] + offset
        with open(fp, 'w') as f:
            json.dump(params, f)

    # 批量并行运行
    execute_entity_task(
        run_constant_climate,
        valid_gdirs,
        nyears=500,
        y0=1650,
        halfsize=30,
        climate_filename='gcm_data',
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=f'_offset{offset}'
    )

    # 收集结果
    success = 0
    for gdir in valid_gdirs:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=f'_offset{offset}')
            if os.path.exists(fp):
                ds = xr.open_dataset(fp)
                a0 = ds.area_m2.values[0]
                a1 = ds.area_m2.values[-1]
                change = (a1 - a0) / a0 * 100
                ds.close()
                results[gdir.rgi_id]['offset_changes'][offset] = change
                success += 1
            else:
                results[gdir.rgi_id]['offset_changes'][offset] = None
        except:
            results[gdir.rgi_id]['offset_changes'][offset] = None

    print(f"offset={offset:+.1f}K 完成: {success}/{len(valid_gdirs)}")

# 恢复原参数
for gdir in valid_gdirs:
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    params['temp_bias'] = original_tbs[gdir.rgi_id]
    with open(fp, 'w') as f:
        json.dump(params, f)

# 输出结果
print("\n=== 结果 ===")
print("RGI_ID\t\t\t" + "\t".join([f"{o:+.1f}K" for o in offsets]))
for rgi_id, r in list(results.items())[:30]:
    line = rgi_id
    for o in offsets:
        val = r['offset_changes'].get(o)
        line += f"\t{val:+.1f}%" if val is not None else "\tNA"
    print(line)

In [ ]:
best_offsets = {}
for rgi_id, r in results.items():
    changes = r['offset_changes']
    valid_offsets = []
    valid_changes = []
    for o in offsets:
        if changes.get(o) is not None:
            valid_offsets.append(o)
            valid_changes.append(changes[o])

    if len(valid_offsets) < 2:
        best_offsets[rgi_id] = {'best_offset': None, 'method': 'insufficient_data'}
        continue

    valid_offsets = np.array(valid_offsets)
    valid_changes = np.array(valid_changes)

    sign_changes = np.where(np.diff(np.sign(valid_changes)) != 0)[0]

    if len(sign_changes) > 0:
        idx = sign_changes[0]
        o1, o2 = valid_offsets[idx], valid_offsets[idx + 1]
        c1, c2 = valid_changes[idx], valid_changes[idx + 1]
        best_o = o1 - c1 * (o2 - o1) / (c2 - c1)
        best_offsets[rgi_id] = {'best_offset': best_o, 'method': 'interpolated'}
    else:
        min_idx = np.argmin(np.abs(valid_changes))
        best_offsets[rgi_id] = {'best_offset': valid_offsets[min_idx], 'method': 'min_abs'}

# 统计
interpolated = [r['best_offset'] for r in best_offsets.values() if r['method'] == 'interpolated' and r['best_offset'] is not None]
min_abs = [r['best_offset'] for r in best_offsets.values() if r['method'] == 'min_abs' and r['best_offset'] is not None]
insufficient = sum(1 for r in best_offsets.values() if r['method'] == 'insufficient_data')

print(f"=== 500年best_offset统计 ===")
print(f"插值找到零点: {len(interpolated)} 个冰川")
print(f"取最小绝对值: {len(min_abs)} 个冰川")
print(f"数据不足: {insufficient}")

if interpolated:
    print(f"\n插值零点统计:")
    print(f"  均值: {np.mean(interpolated):.2f}K")
    print(f"  中位数: {np.median(interpolated):.2f}K")
    print(f"  范围: {min(interpolated):.2f} ~ {max(interpolated):.2f}K")

In [ ]:
# import json
#
# # 保存到working_dir
# output_path = os.path.join(cfg.PATHS['working_dir'], 'lia_best_offsets.json')
# with open(output_path, 'w') as f:
#     json.dump(best_offsets, f, indent=2)
#
# print(f"已保存至: {output_path}")
with open(os.path.join(cfg.PATHS['working_dir'], 'lia_best_offsets.json'), 'r') as f:
    saved_offsets = json.load(f)
valid_offsets = [v['best_offset'] for v in saved_offsets.values() if v['best_offset'] is not None]
print(f"lia_best_offsets均值: {np.mean(valid_offsets):.2f}K")
print(f"lia_best_offsets中位数: {np.median(valid_offsets):.2f}K")

output_path = os.path.join(cfg.PATHS['working_dir'], 'lia_best_offsets_500yr.json')
with open(output_path, 'w') as f:
    json.dump(best_offsets, f, indent=2)
print(f"已保存: {output_path}")
###############################################################
# lia_best_offsets均值: -1.91K
# lia_best_offsets中位数: -2.01K

In [ ]:
# 保存变量
# import json
# import os
#
# working_dir = cfg.PATHS['working_dir']
#
# # 1. original_tbs 已保存
#
# # 2. best_offsets 已保存
#
# # 3. 保存valid_gdirs的rgi_id列表
# valid_ids = [gdir.rgi_id for gdir in valid_gdirs]
# with open(os.path.join(working_dir, 'valid_gdirs_ids.json'), 'w') as f:
#     json.dump(valid_ids, f)
#
# # 4. 保存results
# with open(os.path.join(working_dir, 'offset_results.json'), 'w') as f:
#     json.dump(results, f, indent=2)
#
# print("已保存:")
# print(f"  - original_temp_bias.json")
# print(f"  - lia_best_offsets.json")
# print(f"  - valid_gdirs_ids.json")
# print(f"  - offset_results.json")
#
# ############################
# 恢复变量
# with open(os.path.join(working_dir, 'original_temp_bias.json'), 'r') as f:
#     original_tbs = json.load(f)
# with open(os.path.join(working_dir, 'lia_best_offsets.json'), 'r') as f:
#     best_offsets = json.load(f)
# with open(os.path.join(working_dir, 'valid_gdirs_ids.json'), 'r') as f:
#     valid_ids = json.load(f)
#     valid_gdirs = [gdir for gdir in gdirs_run if gdir.rgi_id in valid_ids]

In [ ]:
# output_path = os.path.join(cfg.PATHS['working_dir'], 'original_temp_bias.json')
# with open(output_path, 'w') as f:
#     json.dump(original_tbs, f, indent=2)
# print(f"已保存: {output_path}")


# 3. 保存valid_gdirs的rgi_id列表
valid_ids = [gdir.rgi_id for gdir in valid_gdirs]
with open(os.path.join(working_dir, 'valid_gdirs_ids.json'), 'w') as f:
    json.dump(valid_ids, f)

# 4. 保存results
with open(os.path.join(working_dir, 'offset_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print("保存完成")

In [ ]:
best_offsets = {}
for rgi_id, r in results.items():
    changes = r['offset_changes']
    valid_offsets = []
    valid_changes = []
    for o in offsets:
        if changes.get(o) is not None:
            valid_offsets.append(o)
            valid_changes.append(changes[o])

    if len(valid_offsets) < 2:
        best_offsets[rgi_id] = {'best_offset': None, 'method': 'insufficient_data'}
        continue

    valid_offsets = np.array(valid_offsets)
    valid_changes = np.array(valid_changes)

    sign_changes = np.where(np.diff(np.sign(valid_changes)) != 0)[0]

    if len(sign_changes) > 0:
        idx = sign_changes[0]
        o1, o2 = valid_offsets[idx], valid_offsets[idx + 1]
        c1, c2 = valid_changes[idx], valid_changes[idx + 1]
        best_o = o1 - c1 * (o2 - o1) / (c2 - c1)
        best_offsets[rgi_id] = {'best_offset': best_o, 'method': 'interpolated'}
    else:
        min_idx = np.argmin(np.abs(valid_changes))
        best_offsets[rgi_id] = {'best_offset': valid_offsets[min_idx], 'method': 'min_abs'}

print(f"计算完成: {len(best_offsets)} 个冰川")

In [ ]:
# 验证当前参数
test_tbs = []
for gdir in valid_gdirs[:10]:
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    test_tbs.append(params['temp_bias'])

print(f"当前temp_bias示例: {test_tbs}")
print(f"均值: {np.mean(test_tbs):.2f}K")
print(f"预期: 约0K（原始参数均值-0.05K）")

import json
import os
from oggm.core.flowline import run_constant_climate, FluxBasedModel
from oggm.workflow import execute_entity_task
class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)# 需要放在 cfg 之前 ，不然会报错/卡住

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 22
cfg.PARAMS['continue_on_error'] = True

# 1. 筛选有效冰川
valid_gdirs_lia = []
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    if rgi_id in best_offsets and best_offsets[rgi_id]['best_offset'] is not None:
        valid_gdirs_lia.append(gdir)
print(f"有效冰川数: {len(valid_gdirs_lia)}")

# 2. 备份当前参数（以防original_tbs丢失）
backup_tbs = {}
for gdir in valid_gdirs_lia:
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    backup_tbs[gdir.rgi_id] = params['temp_bias']

# 3. 设置最优offset
for gdir in valid_gdirs_lia:
    rgi_id = gdir.rgi_id
    best_o = best_offsets[rgi_id]['best_offset']
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    params['temp_bias'] = backup_tbs[rgi_id] + best_o
    with open(fp, 'w') as f:
        json.dump(params, f)

# 4. 运行LIA spinup
execute_entity_task(
    run_constant_climate,
    valid_gdirs_lia,
    nyears=500,
    y0=1650,
    halfsize=30,
    climate_filename='gcm_data',
    mb_model_class=MyMBModel,
    evolution_model=FluxBasedModel,
    output_filesuffix='_lia_spinup_500yr',
    store_model_geometry=True
)
print("Spinup完成")

# 5. 恢复原始参数
for gdir in valid_gdirs_lia:
    rgi_id = gdir.rgi_id
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    params['temp_bias'] = backup_tbs[rgi_id]
    with open(fp, 'w') as f:
        json.dump(params, f)
print("已恢复原始temp_bias")

In [ ]:
import json
import os
import numpy as np
import xarray as xr
import pickle
from oggm.core.flowline import run_constant_climate, run_from_climate_data, FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import cfg

# ============================================================
# 配置
# ============================================================
candidate_t0s = [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]
offsets = [0, -0.5, -1, -1.5, -2, -2.5, -3, -3.5, -4]
spinup_years = 500
halfsize = 30

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 22
cfg.PARAMS['continue_on_error'] = True

# ============================================================
# 加载原始参数
# ============================================================
original_tbs = {}
for gdir in valid_gdirs_lia:
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    original_tbs[gdir.rgi_id] = params['temp_bias']
print(f"加载 {len(original_tbs)} 条冰川的原始参数")

# ============================================================
# 存储所有结果
# ============================================================
all_results = {
    'offset_experiments': {},  # 每个t₀的offset实验结果
    'best_offsets': {},        # 每个t₀的最优offset
    'forward_runs': {}         # 每个t₀的正演结果
}

# ============================================================
# 主循环：遍历每个候选 t₀
# ============================================================
for t0 in candidate_t0s:
    print(f"\n{'='*60}")
    print(f"处理 t₀ = {t0}")
    print(f"{'='*60}")

    # ----------------------------------------------------------
    # Step 1: Offset实验，寻找最优参数
    # ----------------------------------------------------------
    offset_results = {gdir.rgi_id: {'offset_changes': {}} for gdir in valid_gdirs_lia}

    for offset in offsets:
        print(f"\n  Offset实验: t₀={t0}, offset={offset:+.1f}K")

        # 设置参数
        for gdir in valid_gdirs_lia:
            rgi_id = gdir.rgi_id
            fp = os.path.join(gdir.dir, 'mb_calib.json')
            with open(fp, 'r') as f:
                params = json.load(f)
            params['temp_bias'] = original_tbs[rgi_id] + offset
            with open(fp, 'w') as f:
                json.dump(params, f)

        # Spinup
        execute_entity_task(
            run_constant_climate,
            valid_gdirs_lia,
            nyears=spinup_years,
            y0=t0,
            halfsize=halfsize,
            climate_filename='gcm_data',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            output_filesuffix=f'_t0_{t0}_offset{offset}'
        )

        # 提取面积变化
        for gdir in valid_gdirs_lia:
            try:
                fp = gdir.get_filepath('model_diagnostics', filesuffix=f'_t0_{t0}_offset{offset}')
                with xr.open_dataset(fp) as ds:
                    area = ds['area_m2'].values
                    change = (area[-1] - area[0]) / area[0] * 100
                    offset_results[gdir.rgi_id]['offset_changes'][offset] = change
            except:
                pass

    all_results['offset_experiments'][t0] = offset_results

    # ----------------------------------------------------------
    # Step 2: 插值找每条冰川的best_offset
    # ----------------------------------------------------------
    best_offsets_t0 = {}
    for rgi_id, data in offset_results.items():
        changes = data['offset_changes']
        if len(changes) < 2:
            best_offsets_t0[rgi_id] = {'best_offset': None}
            continue

        # 排序
        sorted_offsets = sorted(changes.keys())
        sorted_changes = [changes[o] for o in sorted_offsets]

        # 寻找零点
        best_offset = None
        for i in range(len(sorted_changes) - 1):
            if sorted_changes[i] * sorted_changes[i+1] <= 0:
                # 线性插值
                o1, o2 = sorted_offsets[i], sorted_offsets[i+1]
                c1, c2 = sorted_changes[i], sorted_changes[i+1]
                if c2 != c1:
                    best_offset = o1 + (0 - c1) * (o2 - o1) / (c2 - c1)
                else:
                    best_offset = (o1 + o2) / 2
                break

        # 如果没找到零点，取变化最小的
        if best_offset is None:
            min_idx = np.argmin(np.abs(sorted_changes))
            best_offset = sorted_offsets[min_idx]

        best_offsets_t0[rgi_id] = {'best_offset': best_offset}

    all_results['best_offsets'][t0] = best_offsets_t0

    # 统计
    valid_offsets = [v['best_offset'] for v in best_offsets_t0.values() if v['best_offset'] is not None]
    print(f"\n  t₀={t0} best_offset统计:")
    print(f"    有效冰川: {len(valid_offsets)}")
    print(f"    均值: {np.mean(valid_offsets):.2f}K")
    print(f"    中位数: {np.median(valid_offsets):.2f}K")

    # ----------------------------------------------------------
    # Step 3: 用best_offset spinup到平衡态
    # ----------------------------------------------------------
    print(f"\n  用best_offset spinup到平衡态...")

    valid_gdirs_t0 = []
    for gdir in valid_gdirs_lia:
        rgi_id = gdir.rgi_id
        if best_offsets_t0[rgi_id]['best_offset'] is not None:
            valid_gdirs_t0.append(gdir)
            # 设置参数
            fp = os.path.join(gdir.dir, 'mb_calib.json')
            with open(fp, 'r') as f:
                params = json.load(f)
            params['temp_bias'] = original_tbs[rgi_id] + best_offsets_t0[rgi_id]['best_offset']
            with open(fp, 'w') as f:
                json.dump(params, f)

    execute_entity_task(
        run_constant_climate,
        valid_gdirs_t0,
        nyears=spinup_years,
        y0=t0,
        halfsize=halfsize,
        climate_filename='gcm_data',
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        store_model_geometry=True,
        output_filesuffix=f'_spinup_best_t0_{t0}'
    )

    # ----------------------------------------------------------
    # Step 4: 从平衡态正演到2000
    # ----------------------------------------------------------
    print(f"\n  正演 {t0} → 2000...")

    execute_entity_task(
        run_from_climate_data,
        valid_gdirs_t0,
        min_ys=t0,
        max_ys=2000,
        climate_filename='gcm_data',
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=f'_spinup_best_t0_{t0}',
        store_model_geometry=True,
        output_filesuffix=f'_forward_t0_{t0}'
    )

    # ----------------------------------------------------------
    # Step 5: 提取正演结果
    # ----------------------------------------------------------
    forward_results = []
    for gdir in valid_gdirs_t0:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=f'_forward_t0_{t0}')
            with xr.open_dataset(fp) as ds:
                forward_results.append({
                    'rgi_id': gdir.rgi_id,
                    'time': ds['time'].values,
                    'area': ds['area_m2'].values,
                    'volume': ds['volume_m3'].values,
                    'length': ds['length_m'].values,
                })
        except Exception as e:
            print(f"    提取失败: {gdir.rgi_id}")

    all_results['forward_runs'][t0] = forward_results
    print(f"  t₀={t0} 完成，成功提取 {len(forward_results)} 条冰川")

# ============================================================
# 恢复原始参数
# ============================================================
for gdir in valid_gdirs_lia:
    rgi_id = gdir.rgi_id
    fp = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fp, 'r') as f:
        params = json.load(f)
    params['temp_bias'] = original_tbs[rgi_id]
    with open(fp, 'w') as f:
        json.dump(params, f)
print("\n已恢复原始参数")

# ============================================================
# 保存所有结果
# ============================================================
with open('all_results_t0_optimization.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print("结果已保存到 all_results_t0_optimization.pkl")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# 加载结果
with open('all_results_t0_optimization.pkl', 'rb') as f:
    all_results = pickle.load(f)

candidate_t0s = [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]

# ============================================================
# 1. 统计每个t₀的best_offset分布
# ============================================================
print("=" * 60)
print("各t₀的best_offset统计")
print("=" * 60)

for t0 in candidate_t0s:
    offsets = all_results['best_offsets'][t0]
    valid = [v['best_offset'] for v in offsets.values() if v['best_offset'] is not None]
    print("t₀={}: n={}, 均值={:.2f}K, 中位数={:.2f}K, std={:.2f}K".format(
        t0, len(valid), np.mean(valid), np.median(valid), np.std(valid)))

# ============================================================
# 2. 统计每个t₀正演到2000年的冰川数量和面积变化
# ============================================================
print("\n" + "=" * 60)
print("各t₀正演结果统计")
print("=" * 60)

for t0 in candidate_t0s:
    runs = all_results['forward_runs'][t0]
    n_glaciers = len(runs)

    if n_glaciers == 0:
        print("t₀={}: 无有效结果".format(t0))
        continue

    # 计算从t₀到2000的总面积/体积变化
    area_changes = []
    vol_changes = []

    for r in runs:
        a0, a_end = r['area'][0], r['area'][-1]
        v0, v_end = r['volume'][0], r['volume'][-1]

        if a0 > 0:
            area_changes.append((a_end - a0) / a0 * 100)
        if v0 > 0:
            vol_changes.append((v_end - v0) / v0 * 100)

    print("t₀={}: n={}, 面积变化={:.1f}±{:.1f}%, 体积变化={:.1f}±{:.1f}%".format(
        t0, n_glaciers,
        np.mean(area_changes), np.std(area_changes),
        np.mean(vol_changes), np.std(vol_changes)))

# ============================================================
# 3. 与观测数据对比（如果有的话）
# ============================================================
# 假设你有LIA→现代的观测面积变化（从Lee et al. 2021）
# observed_area_change = -23.4  # 示例值

print("\n" + "=" * 60)
print("模拟vs观测对比")
print("=" * 60)

# 计算每个t₀的平均面积变化，找最接近观测值的
observed_change = -23.4  # 根据你的数据调整

t0_vs_obs = {}
for t0 in candidate_t0s:
    runs = all_results['forward_runs'][t0]
    if len(runs) == 0:
        continue

    area_changes = []
    for r in runs:
        a0, a_end = r['area'][0], r['area'][-1]
        if a0 > 0:
            area_changes.append((a_end - a0) / a0 * 100)

    mean_change = np.mean(area_changes)
    diff = abs(mean_change - observed_change)
    t0_vs_obs[t0] = {'mean_change': mean_change, 'diff_from_obs': diff}
    print("t₀={}: 模拟面积变化={:.1f}%, 与观测差={:.1f}%".format(
        t0, mean_change, diff))

# 找最优t₀
best_t0 = min(t0_vs_obs.keys(), key=lambda x: t0_vs_obs[x]['diff_from_obs'])
print("\n最优t₀ = {} (与观测最接近)".format(best_t0))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

candidate_t0s = [1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]

# ============================================================
# 找出在所有t₀中都有结果的冰川
# ============================================================
common_rgi_ids = None
for t0 in candidate_t0s:
    runs = all_results['forward_runs'][t0]
    rgi_ids = set(r['rgi_id'] for r in runs)
    if common_rgi_ids is None:
        common_rgi_ids = rgi_ids
    else:
        common_rgi_ids = common_rgi_ids & rgi_ids

common_rgi_ids = list(common_rgi_ids)
print("在所有t₀中都有结果的冰川: {} 条".format(len(common_rgi_ids)))

# ============================================================
# 选几条冰川绘图
# ============================================================
# 随机选6条，或者你可以指定
np.random.seed(42)
selected_ids = np.random.choice(common_rgi_ids, size=min(6, len(common_rgi_ids)), replace=False)

# 构建数据索引
data_index = {}
for t0 in candidate_t0s:
    data_index[t0] = {r['rgi_id']: r for r in all_results['forward_runs'][t0]}

# ============================================================
# 绘图：每条冰川一个子图，不同t₀用不同颜色
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
colors = plt.cm.viridis(np.linspace(0, 1, len(candidate_t0s)))

for idx, (ax, rgi_id) in enumerate(zip(axes.flat, selected_ids)):
    for i, t0 in enumerate(candidate_t0s):
        r = data_index[t0].get(rgi_id)
        if r is None:
            continue

        time = r['time']
        area = r['area'] / 1e6  # km²

        ax.plot(time, area, color=colors[i], label='t₀={}'.format(t0), linewidth=1.5)

    ax.set_xlabel('Year')
    ax.set_ylabel('Area (km²)')
    ax.set_title(rgi_id)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('不同起始年份(t₀)下的冰川面积演化', fontsize=14)
plt.tight_layout()
plt.savefig('glacier_evolution_multi_t0.png', dpi=150)
plt.show()

# ============================================================
# 绘图：体积演化
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (ax, rgi_id) in enumerate(zip(axes.flat, selected_ids)):
    for i, t0 in enumerate(candidate_t0s):
        r = data_index[t0].get(rgi_id)
        if r is None:
            continue

        time = r['time']
        volume = r['volume'] / 1e9  # km³

        ax.plot(time, volume, color=colors[i], label='t₀={}'.format(t0), linewidth=1.5)

    ax.set_xlabel('Year')
    ax.set_ylabel('Volume (km³)')
    ax.set_title(rgi_id)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('不同起始年份(t₀)下的冰川体积演化', fontsize=14)
plt.tight_layout()
plt.savefig('glacier_volume_multi_t0.png', dpi=150)
plt.show()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# 刚才图中的6条冰川
selected_ids = ['RGI60-15.04491', 'RGI60-15.02806', 'RGI60-15.04378',
                'RGI60-15.02321', 'RGI60-15.03674', 'RGI60-15.05288']

# 构建gdir索引
gdir_dict = {gdir.rgi_id: gdir for gdir in valid_gdirs_lia}

# 10年滑动平均函数
def smooth(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

colors = plt.cm.tab10(np.linspace(0, 1, len(selected_ids)))

# 存储所有冰川的气候数据
all_temp = {}
all_prcp = {}

for i, rgi_id in enumerate(selected_ids):
    gdir = gdir_dict.get(rgi_id)
    if gdir is None:
        continue

    fp = gdir.get_filepath('gcm_data')

    with xr.open_dataset(fp) as ds:
        temp = ds['temp'].values
        prcp = ds['prcp'].values
        time = ds['time'].values

        # 提取年份
        years_monthly = np.array([t.year for t in time])

        # 转为年平均
        unique_years = np.unique(years_monthly)
        temp_annual = []
        prcp_annual = []

        for yr in unique_years:
            mask = years_monthly == yr
            temp_annual.append(temp[mask].mean())
            prcp_annual.append(prcp[mask].sum())

        temp_annual = np.array(temp_annual)
        prcp_annual = np.array(prcp_annual)

        # 10年平滑
        temp_smooth = smooth(temp_annual, 10)
        prcp_smooth = smooth(prcp_annual, 10)
        years_smooth = unique_years[4:-5]

        all_temp[rgi_id] = (years_smooth, temp_smooth)
        all_prcp[rgi_id] = (years_smooth, prcp_smooth)

        # 绘制温度
        axes[0].plot(years_smooth, temp_smooth - 273.15, color=colors[i],
                     linewidth=1.5, label=rgi_id)

        # 绘制降水
        axes[1].plot(years_smooth, prcp_smooth * 12, color=colors[i],
                     linewidth=1.5, label=rgi_id)

axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('温度对比 (10年平滑)')
axes[0].axvline(1900, color='r', linestyle='--', alpha=0.5)
axes[0].legend(loc='best', fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Precip (mm/yr)')
axes[1].set_xlabel('Year')
axes[1].set_title('降水对比 (10年平滑)')
axes[1].axvline(1900, color='r', linestyle='--', alpha=0.5)
axes[1].legend(loc='best', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('不同冰川LMR气候数据对比', fontsize=14)
plt.tight_layout()
plt.savefig('lmr_climate_comparison.png', dpi=150)
plt.show()

# 打印各冰川的气候差异
print("\n各冰川平均气候 (1500-2000):")
print("{:<20} {:>10} {:>12}".format("RGI_ID", "Temp(°C)", "Prcp(mm/yr)"))
print("-" * 45)
for rgi_id in selected_ids:
    if rgi_id in all_temp:
        years, temp = all_temp[rgi_id]
        _, prcp = all_prcp[rgi_id]
        print("{:<20} {:>10.2f} {:>12.1f}".format(
            rgi_id, (temp.mean() - 273.15), prcp.mean() * 12))

In [ ]:
# 检查这6条冰川的气候数据是否来自同一格点
for rgi_id in selected_ids:
    gdir = gdir_dict.get(rgi_id)
    print("{}: 中心坐标 ({:.2f}, {:.2f})".format(
        rgi_id, gdir.cenlat, gdir.cenlon))

In [ ]:
import xarray as xr
import numpy as np

selected_ids = ['RGI60-15.04491', 'RGI60-15.02806', 'RGI60-15.04378',
                'RGI60-15.02321', 'RGI60-15.03674', 'RGI60-15.05288']

gdir_dict = {gdir.rgi_id: gdir for gdir in valid_gdirs_lia}

# 获取t0=1700的冰川响应数据
data_index = {r['rgi_id']: r for r in all_results['forward_runs'][1700]}

print("{:<18} {:>6} {:>6} {:>8} {:>10} {:>12}".format(
    "RGI_ID", "Lat", "Lon", "T(°C)", "P(mm/yr)", "ΔArea(%)"))
print("-" * 65)

for rgi_id in selected_ids:
    gdir = gdir_dict.get(rgi_id)
    fp = gdir.get_filepath('gcm_data')

    with xr.open_dataset(fp) as ds:
        temp_mean = ds['temp'].values.mean()
        prcp_mean = ds['prcp'].values.mean() * 12  # mm/yr

    # 冰川面积变化
    r = data_index.get(rgi_id)
    if r:
        area_change = (r['area'][-1] - r['area'][0]) / r['area'][0] * 100
    else:
        area_change = np.nan

    print("{:<18} {:>6.1f} {:>6.1f} {:>8.1f} {:>10.0f} {:>+12.1f}".format(
        rgi_id, gdir.cenlat, gdir.cenlon, temp_mean, prcp_mean, area_change))

In [ ]:
# 加入best_offset和初始面积
print("{:<18} {:>8} {:>10} {:>12} {:>12}".format(
    "RGI_ID", "T(°C)", "best_off", "init_A(km²)", "ΔArea(%)"))
print("-" * 62)

for rgi_id in selected_ids:
    gdir = gdir_dict.get(rgi_id)
    fp = gdir.get_filepath('gcm_data')

    with xr.open_dataset(fp) as ds:
        temp_mean = ds['temp'].values.mean()

    r = data_index.get(rgi_id)
    best_off = all_results['best_offsets'][1700].get(rgi_id, {}).get('best_offset')

    if r:
        init_area = r['area'][0] / 1e6
        area_change = (r['area'][-1] - r['area'][0]) / r['area'][0] * 100
    else:
        init_area = np.nan
        area_change = np.nan

    print("{:<18} {:>8.1f} {:>10.2f} {:>12.1f} {:>+12.1f}".format(
        rgi_id, temp_mean, best_off if best_off else np.nan, init_area, area_change))

In [ ]:
# 加入海拔信息
print("{:<18} {:>8} {:>8} {:>10} {:>12}".format(
    "RGI_ID", "ref_hgt", "best_off", "T(°C)", "ΔArea(%)"))
print("-" * 60)

for rgi_id in selected_ids:
    gdir = gdir_dict.get(rgi_id)
    fp = gdir.get_filepath('gcm_data')

    with xr.open_dataset(fp) as ds:
        ref_hgt = ds.attrs.get('ref_hgt')
        temp_mean = ds['temp'].values.mean()

    r = data_index.get(rgi_id)
    best_off = all_results['best_offsets'][1700].get(rgi_id, {}).get('best_offset')
    area_change = (r['area'][-1] - r['area'][0]) / r['area'][0] * 100 if r else np.nan

    print("{:<18} {:>8.0f} {:>8.2f} {:>10.1f} {:>+12.1f}".format(
        rgi_id, ref_hgt, best_off if best_off else np.nan, temp_mean, area_change))

In [ ]:
# 对比LIA初始面积 vs RGI现代面积
print("{:<18} {:>12} {:>12} {:>12} {:>12}".format(
    "RGI_ID", "LIA_init", "RGI_area", "LIA/RGI", "ΔArea(%)"))
print("-" * 62)

for rgi_id in selected_ids:
    gdir = gdir_dict.get(rgi_id)
    r = data_index.get(rgi_id)

    if r:
        lia_init = r['area'][0] / 1e6  # spinup后的LIA平衡态面积
        rgi_area = gdir.rgi_area_m2 / 1e6  # RGI现代面积
        ratio = lia_init / rgi_area
        area_change = (r['area'][-1] - r['area'][0]) / r['area'][0] * 100

        print("{:<18} {:>12.2f} {:>12.2f} {:>12.2f} {:>+12.1f}".format(
            rgi_id, lia_init, rgi_area, ratio, area_change))

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

selected_ids = ['RGI60-15.04491', 'RGI60-15.02806', 'RGI60-15.04378',
                'RGI60-15.02321', 'RGI60-15.03674', 'RGI60-15.05288']

gdir_dict = {gdir.rgi_id: gdir for gdir in valid_gdirs_lia}

def smooth(data, window=30):
    return np.convolve(data, np.ones(window)/window, mode='valid')

print("{:<18} {:>10} {:>10} {:>10} {:>12}".format(
    "RGI_ID", "T_1700", "T_1900", "T_2000", "ΔT(1700→2000)"))
print("-" * 65)

fig, ax = plt.subplots(figsize=(12, 6))

for rgi_id in selected_ids:
    gdir = gdir_dict.get(rgi_id)
    fp = gdir.get_filepath('gcm_data')

    with xr.open_dataset(fp) as ds:
        temp = ds['temp'].values
        time = ds['time'].values
        years_monthly = np.array([t.year for t in time])

        # 年平均
        unique_years = np.unique(years_monthly)
        temp_annual = []
        for yr in unique_years:
            mask = years_monthly == yr
            temp_annual.append(temp[mask].mean())
        temp_annual = np.array(temp_annual)

        # 关键时期平均温度
        t_1700 = temp_annual[(unique_years >= 1670) & (unique_years <= 1730)].mean()
        t_1900 = temp_annual[(unique_years >= 1870) & (unique_years <= 1930)].mean()
        t_2000 = temp_annual[(unique_years >= 1970) & (unique_years <= 2000)].mean()

        print("{:<18} {:>10.2f} {:>10.2f} {:>10.2f} {:>+12.2f}".format(
            rgi_id, t_1700, t_1900, t_2000, t_2000 - t_1700))

        # 绘制30年平滑曲线
        temp_smooth = smooth(temp_annual, 30)
        years_smooth = unique_years[14:-15]
        ax.plot(years_smooth, temp_smooth, label=rgi_id, linewidth=1.5)

ax.axvline(1700, color='gray', linestyle='--', alpha=0.5)
ax.axvline(1900, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature (°C)')
ax.set_title('LMR温度趋势 (30年平滑)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(1500, 2000)

plt.tight_layout()
plt.savefig('lmr_temp_trend.png', dpi=150)
plt.show()